In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth scikit-learn
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")

    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install -q scikit-learn

In [ ]:
# =============================================
# CONFIGURATION — adjust before running
# =============================================
HF_MODEL            = "Jaiccc/model_0_streaming_timestamp"
HF_DATASET          = "librocubic/model0-boundary-streaming"

RUN_BASE_COMPARISON = True  # set False to skip the base-model pass (saves ~2x time)
OUTPUT_JSONL        = "model0_eval_results.jsonl"

MAX_NEW_TOKENS = 32    # output is always "<ts>, new/old event" — short
MAX_SEQ_LEN    = 4096
TEMPERATURE    = 0.1

In [ ]:
from unsloth import FastLanguageModel
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

print("Loading fine-tuned model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    token=hf_token,
)
FastLanguageModel.for_inference(model)
print("Model ready.")

In [ ]:
from datasets import load_dataset

print("Loading dataset...")
dataset = load_dataset(HF_DATASET, split="test", token=hf_token)
samples = dataset
print(f"Evaluating on {len(samples)} samples from test split")

# Sanity check
ex = samples[0]
print(f"\nExample output label: '{ex['output']}'")

In [ ]:
import re
import numpy as np
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
)


def predict(instruction, input_data, use_base=False):
    """Run one inference pass. use_base=True disables the LoRA adapter."""
    prompt = (
        f"<|im_start|>user<|im_sep|>{instruction}\n\n{input_data}"
        f"<|im_end|><|im_start|>assistant<|im_sep|>"
    )
    inputs = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        use_cache=True,
        temperature=TEMPERATURE,
        pad_token_id=tokenizer.eos_token_id,
    )
    if use_base:
        with model.disable_adapter():
            out = model.generate(input_ids=inputs, **gen_kwargs)
    else:
        out = model.generate(input_ids=inputs, **gen_kwargs)

    raw = tokenizer.batch_decode(out, clean_up_tokenization_spaces=True)[0]
    m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw, re.S)
    return (m.group(1).strip() if m else raw.split("assistant")[-1].strip()).lower()


def parse_label(text):
    """Returns True='new event', False='old event', None=unparseable."""
    if "new" in text:
        return True
    if "old" in text:
        return False
    return None


def windowdiff(ref, hyp, k):
    """Pevzner & Hearst WindowDiff. Lower = better; 0 = perfect."""
    N = len(ref)
    if N <= k:
        return 0.0
    return sum(
        abs(sum(ref[i + 1:i + k + 1]) - sum(hyp[i + 1:i + k + 1])) > 0
        for i in range(N - k)
    ) / (N - k)


def compute_metrics(y_true, y_pred):
    """Precision, recall, F1 for the 'new event' positive class + WindowDiff."""
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, pos_label=True, average='binary', zero_division=0
    )
    acc = sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[False, True]).ravel()
    n_seg = max(1, sum(y_true))
    k = max(1, len(y_true) // (2 * n_seg))
    wd = windowdiff([int(x) for x in y_true], [int(x) for x in y_pred], k)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    null_acc = (tn + fn) / len(y_true)   # "always predict old" accuracy
    return dict(
        accuracy=acc, precision=float(p), recall=float(r), f1=float(f1),
        tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn), windowdiff=wd,
        balanced_accuracy=float(bal_acc), mcc=float(mcc), null_accuracy=float(null_acc),
    )


def print_metrics(label, m, n):
    print(f"\n{label} ({n} samples)")
    print(f"  Accuracy:         {m['accuracy']:.3f}")
    print(f"  Balanced Acc:     {m['balanced_accuracy']:.3f}")
    print(f"  MCC:              {m['mcc']:.3f}")
    print(f"  Precision:        {m['precision']:.3f}")
    print(f"  Recall:           {m['recall']:.3f}")
    print(f"  F1:               {m['f1']:.3f}")
    print(f"  WindowDiff:       {m['windowdiff']:.3f}")
    print(f"  Confusion:        TP={m['tp']} FP={m['fp']} FN={m['fn']} TN={m['tn']}")
    print(f"  Null baseline:    {m['null_accuracy']:.3f}  (always-predict-old accuracy)")

In [ ]:
import json
from tqdm.notebook import tqdm

ft_records = []
ft_true, ft_pred = [], []
ft_parse_errors = 0

print("Running fine-tuned model inference...")
for i, sample in enumerate(tqdm(samples)):
    expected  = sample['output'].strip().lower()
    predicted = predict(sample['instruction'], sample['input'], use_base=False)

    gt_label   = parse_label(expected)
    pred_label = parse_label(predicted)

    if gt_label is None or pred_label is None:
        ft_parse_errors += 1
        continue

    ft_true.append(gt_label)
    ft_pred.append(pred_label)
    ft_records.append({
        "idx": i,
        "expected": expected,
        "predicted": predicted,
        "correct": gt_label == pred_label,
        "model": "fine_tuned",
    })

if ft_parse_errors:
    print(f"  ({ft_parse_errors} samples skipped — unparseable output)")

ft_metrics = compute_metrics(ft_true, ft_pred)
print_metrics("Fine-Tuned Model", ft_metrics, len(ft_true))

In [ ]:
if RUN_BASE_COMPARISON:
    base_records = []
    base_true, base_pred = [], []
    base_parse_errors = 0

    print("Running base model inference (LoRA disabled)...")
    for i, sample in enumerate(tqdm(samples)):
        expected  = sample['output'].strip().lower()
        predicted = predict(sample['instruction'], sample['input'], use_base=True)

        gt_label   = parse_label(expected)
        pred_label = parse_label(predicted)

        if gt_label is None or pred_label is None:
            base_parse_errors += 1
            continue

        base_true.append(gt_label)
        base_pred.append(pred_label)
        base_records.append({
            "idx": i,
            "expected": expected,
            "predicted": predicted,
            "correct": gt_label == pred_label,
            "model": "base",
        })

    if base_parse_errors:
        print(f"  ({base_parse_errors} samples skipped — unparseable output)")

    base_metrics = compute_metrics(base_true, base_pred)
    print_metrics("Base Model", base_metrics, len(base_true))

    # ---- Side-by-side comparison table ----
    metrics_to_show  = ['accuracy', 'balanced_accuracy', 'mcc', 'precision', 'recall', 'f1', 'windowdiff']
    higher_is_better = {
        'accuracy': True, 'balanced_accuracy': True, 'mcc': True,
        'precision': True, 'recall': True,
        'f1': True, 'windowdiff': False,
    }

    print("\n" + "=" * 58)
    print(f"{'Metric':<18} {'Fine-Tuned':>12} {'Base Model':>12} {'Delta':>10}")
    print("=" * 58)
    for metric in metrics_to_show:
        ft_val   = ft_metrics[metric]
        base_val = base_metrics[metric]
        delta    = ft_val - base_val
        sign     = "+" if delta >= 0 else ""
        better   = (delta > 0) == higher_is_better[metric]
        marker   = " <" if (better and delta != 0) else ""
        print(f"{metric:<18} {ft_val:>12.3f} {base_val:>12.3f} {sign}{delta:>9.3f}{marker}")
    print("=" * 58)
    print("< = fine-tuned is better")
    print(f"  Null baseline (always-predict-old): {ft_metrics['null_accuracy']:.3f}")
else:
    print("Base comparison skipped (RUN_BASE_COMPARISON=False).")
    base_records = []

In [ ]:
# Save all inference records for offline analysis
all_records = ft_records + base_records
with open(OUTPUT_JSONL, "w") as f:
    for r in all_records:
        f.write(json.dumps(r) + "\n")
print(f"Saved {len(all_records)} records to {OUTPUT_JSONL}")

from google.colab import files
files.download(OUTPUT_JSONL)